<a href="https://colab.research.google.com/github/ReservaAraras/ECO-SIM/blob/main/SVGtoGIF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# @title Converter SVG para GIF no Google Drive
# 1. Instalação das dependências necessárias (ocultando os logs)
!apt-get update > /dev/null 2>&1
!apt-get install -y libcairo2-dev > /dev/null 2>&1
!pip install cairosvg google-api-python-client > /dev/null 2>&1

import os
import io
import cairosvg
from PIL import Image
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

# 2. Autenticação no Google Drive
print("Solicitando autenticação do Google Drive...")
auth.authenticate_user()
drive_service = build('drive', 'v3')

# ID da pasta extraído da sua URL
folder_id = '1t6uOtPiF0Fh6Utbgt4xq-LWSxC7BlsKC'

# 3. Buscar todos os arquivos na pasta
print("Buscando arquivos na pasta...")
query = f"'{folder_id}' in parents and trashed=false"
items =[]
page_token = None

# Paginação caso a pasta tenha muitos arquivos
while True:
    results = drive_service.files().list(
        q=query,
        fields="nextPageToken, files(id, name, mimeType)",
        pageToken=page_token,
        pageSize=1000
    ).execute()
    items.extend(results.get('files',[]))
    page_token = results.get('nextPageToken')
    if not page_token:
        break

# Filtrar apenas os arquivos SVG
svg_files =[f for f in items if f['name'].lower().endswith('.svg') or f.get('mimeType') == 'image/svg+xml']

if not svg_files:
    print('Nenhum arquivo SVG encontrado na pasta.')
else:
    print(f'{len(svg_files)} arquivo(s) SVG encontrado(s). Iniciando conversão...\n')

    for item in svg_files:
        file_id = item['id']
        file_name = item['name']
        print(f'Processando: {file_name}')

        try:
            # 4. Download do arquivo SVG para a memória
            request = drive_service.files().get_media(fileId=file_id)
            svg_content = io.BytesIO()
            downloader = MediaIoBaseDownload(svg_content, request)
            done = False
            while done is False:
                status, done = downloader.next_chunk()

            # 5. Converter SVG para PNG (em memória)
            # O formato GIF não é suportado diretamente pelo cairosvg, então passamos por PNG
            png_content = cairosvg.svg2png(bytestring=svg_content.getvalue())

            # 6. Converter PNG para GIF usando Pillow (PIL)
            png_image = Image.open(io.BytesIO(png_content))

            # Tratamento de transparência: cria um fundo branco para evitar bugs visuais no GIF
            if png_image.mode in ('RGBA', 'LA') or (png_image.mode == 'P' and 'transparency' in png_image.info):
                background = Image.new('RGB', png_image.size, (255, 255, 255))
                rgba_image = png_image.convert('RGBA')
                background.paste(rgba_image, mask=rgba_image.split()[3]) # Usa o canal alpha como máscara
                gif_image = background
            else:
                gif_image = png_image.convert('RGB')

            # Define o nome e caminho temporário do GIF
            gif_filename = file_name.rsplit('.', 1)[0] + '.gif'
            gif_path = f'/content/{gif_filename}'

            # Salva o GIF localmente no Colab
            gif_image.save(gif_path, format='GIF')

            # 7. Upload do GIF de volta para a mesma pasta no Drive
            file_metadata = {'name': gif_filename, 'parents': [folder_id]}
            media = MediaFileUpload(gif_path, mimetype='image/gif')
            drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()

            print(f'  -> Salvo com sucesso: {gif_filename}')

            # Limpa o arquivo local temporário
            os.remove(gif_path)

        except Exception as e:
            print(f'  -> Erro ao processar {file_name}: {e}')

print('\nProcesso concluído!')

Solicitando autenticação do Google Drive...


Buscando arquivos na pasta...


92 arquivo(s) SVG encontrado(s). Iniciando conversão...

Processando: notebook_80.svg
  -> Salvo com sucesso: notebook_80.gif
Processando: notebook_75.svg
  -> Salvo com sucesso: notebook_75.gif
Processando: notebook_78.svg
  -> Salvo com sucesso: notebook_78.gif
Processando: notebook_79.svg
  -> Salvo com sucesso: notebook_79.gif
Processando: notebook_74.svg
  -> Salvo com sucesso: notebook_74.gif
Processando: notebook_77.svg
  -> Salvo com sucesso: notebook_77.gif
Processando: notebook_76.svg
  -> Salvo com sucesso: notebook_76.gif
Processando: notebook_73.svg
  -> Salvo com sucesso: notebook_73.gif
Processando: notebook_66.svg
  -> Salvo com sucesso: notebook_66.gif
Processando: notebook_72.svg
  -> Salvo com sucesso: notebook_72.gif
Processando: notebook_71.svg
  -> Salvo com sucesso: notebook_71.gif
Processando: notebook_70.svg
  -> Salvo com sucesso: notebook_70.gif
Processando: notebook_69.svg
  -> Salvo com sucesso: notebook_69.gif
Processando: notebook_68.svg
  -> Salvo com su